# Palace CPW Simulation — Lumped Ports

[Palace](https://awslabs.github.io/palace/) is an open-source 3D electromagnetic simulator supporting eigenmode, driven (S-parameter), and electrostatic simulations. This notebook demonstrates using the `gsim.palace` API to run a driven simulation on an asymmetric CPW (coplanar waveguide) structure.

**Requirements:**

- IHP PDK: `uv pip install ihp-gdsfactory`
- [GDSFactory+](https://gdsfactory.com) account for cloud simulation

### Build component

In [ ]:
import gdsfactory as gf
from ihp import LAYER, PDK

PDK.activate()

METAL_LAYER = LAYER.Metal1drawing
BRIDGE_LAYER = LAYER.Metal2drawing
VIA_LAYER = LAYER.Via1drawing


@gf.cell
def asymmetric_cpw_with_airbridges(
    # launch pad / input discontinuity (symmetric)
    pad_length: float = 60,
    s_pad_width: float = 30,
    g_pad_width: float = 50,
    gap_pad: float = 15,
    taper_length: float = 80,
    # main line (asymmetric ground widths, constant gap so tapers stay parallel)
    line_length: float = 300,
    s_width: float = 20,
    g_width_top: float = 40,
    g_width_bot: float = 20,
    gap_line: float = 15,  # keep equal to gap_pad -> parallel tapers
    # airbridges
    with_airbridges: bool = True,
    num_bridges: int = 2,
    bridge_width: float = 10,
    bridge_spacing: float = 100,
    bridge_start_offset: float = 5,
    via_margin: float = 4,
    metal_layer=METAL_LAYER,
    bridge_layer=BRIDGE_LAYER,
    via_layer=VIA_LAYER,
) -> gf.Component:
    """Launch pad -> taper -> asymmetric-width CPW line with airbridges -> o2."""
    c = gf.Component()

    if gap_line != gap_pad:
        print("Warning: gap_line != gap_pad. The taper gaps will NOT be parallel.")
    if g_width_top >= g_pad_width or g_width_bot >= g_pad_width:
        raise ValueError(
            f"g_width_top ({g_width_top}) and g_width_bot ({g_width_bot}) "
            f"must both be < g_pad_width ({g_pad_width}), or that ground "
            "won't visibly neck down in width along the taper."
        )

    # Signal (center) trace
    pad_pts = [
        (-pad_length, -s_pad_width / 2),
        (-pad_length, s_pad_width / 2),
        (0, s_pad_width / 2),
        (0, -s_pad_width / 2),
    ]
    c.add_polygon(pad_pts, layer=metal_layer)

    s_taper_pts = [
        (0, s_pad_width / 2),
        (taper_length, s_width / 2),
        (taper_length, -s_width / 2),
        (0, -s_pad_width / 2),
    ]
    c.add_polygon(s_taper_pts, layer=metal_layer)

    line_pts = [
        (taper_length, -s_width / 2),
        (taper_length, s_width / 2),
        (taper_length + line_length, s_width / 2),
        (taper_length + line_length, -s_width / 2),
    ]
    c.add_polygon(line_pts, layer=metal_layer)

    # Ground traces (top & bottom), asymmetric in widths only
    def draw_ground(is_top: bool):
        sign = 1 if is_top else -1
        g_width_line = g_width_top if is_top else g_width_bot

        y_pad_inner = sign * (s_pad_width / 2 + gap_pad)
        y_pad_outer = sign * (s_pad_width / 2 + gap_pad + g_pad_width)
        y_line_inner = sign * (s_width / 2 + gap_line)
        y_line_outer = sign * (s_width / 2 + gap_line + g_width_line)

        pad_pts = [
            (-pad_length, y_pad_inner),
            (-pad_length, y_pad_outer),
            (0, y_pad_outer),
            (0, y_pad_inner),
        ]
        c.add_polygon(pad_pts, layer=metal_layer)

        taper_pts = [
            (0, y_pad_inner),
            (0, y_pad_outer),
            (taper_length, y_line_outer),
            (taper_length, y_line_inner),
        ]
        c.add_polygon(taper_pts, layer=metal_layer)

        line_pts = [
            (taper_length, y_line_inner),
            (taper_length, y_line_outer),
            (taper_length + line_length, y_line_outer),
            (taper_length + line_length, y_line_inner),
        ]
        c.add_polygon(line_pts, layer=metal_layer)

        return sign * (s_width / 2 + gap_line + g_width_line / 2)

    y_center_top = draw_ground(is_top=True)
    y_center_bot = draw_ground(is_top=False)

    # Airbridges: strap G_top to G_bottom across signal + both gaps
    if with_airbridges:
        bridge_start_x = taper_length + bridge_start_offset
        for i in range(num_bridges):
            bx = bridge_start_x + i * bridge_spacing

            strap_pts = [
                (bx - bridge_width / 2, y_center_bot),
                (bx - bridge_width / 2, y_center_top),
                (bx + bridge_width / 2, y_center_top),
                (bx + bridge_width / 2, y_center_bot),
            ]
            c.add_polygon(strap_pts, layer=bridge_layer)

            via_size_top = min(bridge_width, g_width_top) - via_margin
            via_top_pts = [
                (bx - via_size_top / 2, y_center_top - via_size_top / 2),
                (bx - via_size_top / 2, y_center_top + via_size_top / 2),
                (bx + via_size_top / 2, y_center_top + via_size_top / 2),
                (bx + via_size_top / 2, y_center_top - via_size_top / 2),
            ]
            c.add_polygon(via_top_pts, layer=via_layer)

            via_size_bot = min(bridge_width, g_width_bot) - via_margin
            via_bot_pts = [
                (bx - via_size_bot / 2, y_center_bot - via_size_bot / 2),
                (bx - via_size_bot / 2, y_center_bot + via_size_bot / 2),
                (bx + via_size_bot / 2, y_center_bot + via_size_bot / 2),
                (bx + via_size_bot / 2, y_center_bot - via_size_bot / 2),
            ]
            c.add_polygon(via_bot_pts, layer=via_layer)

    # Ports
    c.add_port(
        name="o1",
        center=(-pad_length, 0),
        width=s_pad_width,
        orientation=180,
        port_type="electrical",
        layer=metal_layer,
    )
    c.add_port(
        name="o2",
        center=(taper_length + line_length, 0),
        width=s_width,
        orientation=0,
        port_type="electrical",
        layer=metal_layer,
    )
    return c

### With airbridges

In [ ]:
port_kwargs = dict(s_pad_width=30, gap_pad=15)
c_with = asymmetric_cpw_with_airbridges(with_airbridges=True, **port_kwargs)
cc_with = c_with.copy()
cc_with.draw_ports()
cc_with.plot()

### Configure and run simulation with DrivenSim

In [ ]:
from gsim.common.stack import get_stack
from gsim.palace import DrivenSim


def run_case(mode: int, tag: str):
    # Create simulation object
    sim = DrivenSim()

    # Set output directory
    sim.set_output_dir(f"./palace-sim-asymmetric-cpw-with-airbridges-{tag}")

    # Set the component geometry
    sim.set_geometry(c_with)

    # Configure layer stack from active PDK
    stack = get_stack()
    sim.set_stack(stack)
    sim.set_airbox(margin_x=0.0, margin_y=25.0, z_above=50.0, z_below=50.0)

    # Force the vertical Via1 pillars to be perfect conductors
    sim.add_pec(gds_layer=(19, 0), from_layer="metal1", to_layer="metal2")

    # o1: lumped port at the symmetric launch pad -> excites CPW mode only
    sim.add_cpw_port(
        "o1",
        layer="metal1",
        s_width=port_kwargs["s_pad_width"],
        gap_width=port_kwargs["gap_pad"],
        excited=True,
    )
    # o2: 2-mode wave port at the asymmetric line's own cross-section
    sim.add_wave_port(
        "o2",
        layer="metal1",
        mode=mode,
        max_size=True,
        excited=False,
    )

    # Configure driven simulation (frequency sweep for S-parameters)
    sim.set_driven(fmin=1e9, fmax=100e9, num_points=300)

    # Validate configuration
    print(sim.validate_config())

    sim.mesh(
        preset="default", refined_mesh_size=2.0, max_mesh_size=40, fmax=150
    )  # keep meshing params identical across runs
    return sim


sim_cpw_with = run_case(mode=1, tag="mode1")
sim_slot_with = run_case(mode=2, tag="mode2")

In [ ]:
sim_cpw_with.plot_mesh(show_groups=["metal", "P", "via"], interactive=True)

In [ ]:
sim_cpw_with.plot_mesh(
    style="solid",
    transparent_groups=["air__None", "SiO2__None", "SiO2__passive", "air__passive"],
    interactive=True,
)

### Run simulation

In [ ]:
# results = sim.run_local(palace_executable="~/palace/build/bin/palace")
results_cpw_with = sim_cpw_with.run()
results_slot_with = sim_slot_with.run()

### Plot S-parameters

In [ ]:
results_cpw_with.plot()
results_slot_with.plot()

In [ ]:
import matplotlib.pyplot as plt

freqs = results_cpw_with.freq  # confirm units — likely Hz, so /1e9 for GHz
ratio_db = results_slot_with.s21.db - results_cpw_with.s21.db

plt.plot(results_cpw_with.freq, ratio_db)
plt.xlabel("Frequency (GHz)")
plt.ylabel("Slotline/CPW ratio (dB)")
plt.title("Mode conversion at asymmetric CPW cross-section")
plt.grid(True)
plt.show()

### Without air-bridges

In [ ]:
c_without = asymmetric_cpw_with_airbridges(with_airbridges=False, **port_kwargs)
cc_without = c_without.copy()
cc_without.draw_ports()
cc_without.plot()

### Configure and run simulation with DrivenSim

In [ ]:
def run_case_without(mode: int, tag: str):
    # Create simulation object
    sim = DrivenSim()

    # Set output directory
    sim.set_output_dir(f"./palace-sim-asymmetric-cpw-without-airbridges-{tag}")

    # Set the component geometry
    sim.set_geometry(c_without)

    # Configure layer stack from active PDK
    stack = get_stack()
    sim.set_stack(stack)
    sim.set_airbox(margin_x=0.0, margin_y=25.0, z_above=50.0, z_below=50.0)

    # o1: lumped port at the symmetric launch pad -> excites CPW mode only
    sim.add_cpw_port(
        "o1",
        layer="metal1",
        s_width=port_kwargs["s_pad_width"],
        gap_width=port_kwargs["gap_pad"],
        excited=True,
    )
    # o2: 2-mode wave port at the asymmetric line's own cross-section
    sim.add_wave_port(
        "o2",
        layer="metal1",
        mode=mode,
        max_size=True,
        excited=False,
    )

    # Configure driven simulation (frequency sweep for S-parameters)
    sim.set_driven(fmin=1e9, fmax=100e9, num_points=300)

    # Validate configuration
    print(sim.validate_config())

    sim.mesh(
        preset="default", refined_mesh_size=2.0, max_mesh_size=40, fmax=150
    )  # keep meshing params identical across runs
    return sim


sim_cpw_without = run_case_without(mode=1, tag="mode1")
sim_slot_without = run_case_without(mode=2, tag="mode2")

### Run simulation

In [ ]:
results_cpw_without = sim_cpw_without.run()
results_slot_without = sim_slot_without.run()

### Plot S-parameters

In [ ]:
results_cpw_without.plot()
results_slot_without.plot()

In [ ]:
import matplotlib.pyplot as plt

freqs = results_cpw_without.freq  # confirm units — likely Hz, so /1e9 for GHz
ratio_db = results_slot_without.s21.db - results_cpw_without.s21.db

plt.plot(results_cpw_without.freq, ratio_db)
plt.xlabel("Frequency (GHz)")
plt.ylabel("Slotline/CPW ratio (dB)")
plt.title("Mode conversion at asymmetric CPW cross-section without airbridges")
plt.grid(True)
plt.show()

In [ ]:
plt.plot(
    results_cpw_with.freq,
    results_slot_with.s21.db - results_cpw_with.s21.db,
    label="with bridges",
)
plt.plot(
    results_cpw_without.freq,
    results_slot_without.s21.db - results_cpw_without.s21.db,
    label="without bridges",
)
plt.xlabel("Frequency (GHz)")
plt.ylabel("Slotline/CPW ratio (dB)")
plt.legend()
plt.grid(True)

In [ ]:
ratio_with = results_slot_with.s21.db - results_cpw_with.s21.db
ratio_without = results_slot_without.s21.db - results_cpw_without.s21.db

plt.plot(results_cpw_with.freq, ratio_with - ratio_without)
plt.xlabel("Frequency (GHz)")
plt.ylabel("Bridge suppression (dB)")
plt.grid(True)

### Test

In [ ]:
@gf.cell
def cpw_cross_section_for_es(
    length: float = 20,
    s_width: float = 20,
    g_width_top: float = 40,
    g_width_bot: float = 20,
    gap: float = 15,
    s_layer=LAYER.Metal1drawing,
    g1_layer=LAYER.Metal2drawing,
    g2_layer=LAYER.Metal3drawing,
) -> gf.Component:
    c = gf.Component()

    c.add_polygon(
        [
            (0, -s_width / 2),
            (0, s_width / 2),
            (length, s_width / 2),
            (length, -s_width / 2),
        ],
        layer=s_layer,
    )

    y_g1_in = s_width / 2 + gap
    y_g1_out = y_g1_in + g_width_top
    c.add_polygon(
        [(0, y_g1_in), (0, y_g1_out), (length, y_g1_out), (length, y_g1_in)],
        layer=g1_layer,
    )

    y_g2_in = -(s_width / 2 + gap)
    y_g2_out = y_g2_in - g_width_bot
    c.add_polygon(
        [(0, y_g2_in), (0, y_g2_out), (length, y_g2_out), (length, y_g2_in)],
        layer=g2_layer,
    )

    c.add_port(
        name="s",
        center=(length / 2, 0),
        width=s_width,
        orientation=90,
        port_type="electrical",
        layer=s_layer,
    )
    c.add_port(
        name="g1",
        center=(length / 2, y_g1_in + g_width_top / 2),
        width=g_width_top,
        orientation=90,
        port_type="electrical",
        layer=g1_layer,
    )
    c.add_port(
        name="g2",
        center=(length / 2, y_g2_in - g_width_bot / 2),
        width=g_width_bot,
        orientation=90,
        port_type="electrical",
        layer=g2_layer,
    )

    return c

In [ ]:
from gsim.palace import ElectrostaticSim
from gsim.common.stack import get_stack
import json

# Force a definitely-new component instance (different length -> different cache hash)
c_test = cpw_cross_section_for_es(length=21)
print(c_test.name)  # sanity check: should be a different name than before

sim_es = ElectrostaticSim()
sim_es.set_output_dir("./palace-sim-es-cross-section")
sim_es.set_geometry(c_test)
sim_es.set_stack(get_stack())
sim_es.set_airbox(margin_x=0.0, margin_y=25.0, z_above=50.0, z_below=50.0)

sim_es.add_terminal("s", layer="metal1")
sim_es.add_terminal("g1", layer="metal2")
sim_es.add_terminal("g2", layer="metal3")

sim_es.set_electrostatic()
print(sim_es.validate_config())

sim_es.mesh(preset="default")
sim_es.write_config()
results_es = sim_es.run_local(palace_executable="~/palace/build/bin/palace")

In [ ]:
import os

for root, dirs, files in os.walk("palace-sim-es-cross-section/output"):
    for f in files:
        print(os.path.join(root, f))

In [ ]:
import pandas as pd

df = pd.read_csv("palace-sim-es-cross-section/output/palace/terminal-C.csv")
print(df)

In [ ]:
@gf.cell
def via_connectivity_test(
    pad_size: float = 20,
    via_size: float = 8,
    g1_layer=LAYER.Metal1drawing,
    g2_layer=LAYER.Metal2drawing,
    via_layer=LAYER.Via1drawing,
    with_via: bool = True,
) -> gf.Component:
    c = gf.Component()

    # g1: metal1 pad
    c.add_polygon(
        [(0, 0), (0, pad_size), (pad_size, pad_size), (pad_size, 0)], layer=g1_layer
    )
    # g2: metal2 pad, directly overlapping in XY (stacked above g1)
    c.add_polygon(
        [(0, 0), (0, pad_size), (pad_size, pad_size), (pad_size, 0)], layer=g2_layer
    )

    if with_via:
        off = (pad_size - via_size) / 2
        c.add_polygon(
            [
                (off, off),
                (off, off + via_size),
                (off + via_size, off + via_size),
                (off + via_size, off),
            ],
            layer=via_layer,
        )

    c.add_port(
        name="g1",
        center=(pad_size / 2, 0),
        width=pad_size,
        orientation=270,
        port_type="electrical",
        layer=g1_layer,
    )
    c.add_port(
        name="g2",
        center=(pad_size / 2, pad_size),
        width=pad_size,
        orientation=90,
        port_type="electrical",
        layer=g2_layer,
    )

    return c